# Stage 6 & 7: Algorithm 3 — LightGBM Gradient Boosting

**Owner:** Member C  
**Inputs:** `data/final/` — X_train, X_val, y_train, y_val  
**Outputs:** `models/lightgbm_tuned.pkl`  

### Scope
This notebook implements the gradient boosting ensemble model family for late delivery risk prediction.
It trains a baseline LightGBM Classifier, performs 5-Fold Stratified K-Fold hyperparameter tuning on `X_train` (tuning `scale_pos_weight`, `learning_rate`, `num_leaves`), evaluates performance on `X_val`, and exports the tuned model artifact.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import lightgbm as lgb

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    recall_score, precision_score, classification_report
)

FINAL_DIR = Path('../data/final')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_val   = pd.read_csv(FINAL_DIR / 'X_val.csv')
y_train = pd.read_csv(FINAL_DIR / 'y_train.csv').values.ravel()
y_val   = pd.read_csv(FINAL_DIR / 'y_val.csv').values.ravel()

scale_pos_weight_val = (len(y_train) - sum(y_train)) / sum(y_train)
print(f'X_train shape: {X_train.shape}, X_val shape: {X_val.shape}')
print(f'Imbalance scale_pos_weight: {scale_pos_weight_val:.2f}')


In [ ]:
# ── Default LightGBM Baseline ─────────────────────────────────────────────
lgb_base = lgb.LGBMClassifier(random_state=42, verbose=-1)
lgb_base.fit(X_train, y_train)
base_probs = lgb_base.predict_proba(X_val)[:, 1]
print(f'Baseline LightGBM Val ROC-AUC : {roc_auc_score(y_val, base_probs):.4f}')
print(f'Baseline LightGBM Val PR-AUC  : {average_precision_score(y_val, base_probs):.4f}')


In [ ]:
# ── 5-Fold Stratified RandomizedSearchCV ──────────────────────────────────
param_dist = {
    'n_estimators': [150, 300, 450],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [4, 6, 8, 10],
    'num_leaves': [15, 31, 63],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0],
    'scale_pos_weight': [1.0, scale_pos_weight_val / 2.0, scale_pos_weight_val]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rand = RandomizedSearchCV(
    lgb.LGBMClassifier(random_state=42, verbose=-1),
    param_dist, n_iter=12, scoring='average_precision', cv=cv, random_state=42, n_jobs=-1
)
rand.fit(X_train, y_train)

print('Best Hyperparameters:', rand.best_params_)
print(f'Best Stratified CV PR-AUC: {rand.best_score_:.4f}')

best_lgb = rand.best_estimator_
joblib.dump(best_lgb, MODELS_DIR / 'lightgbm_tuned.pkl')
print('Saved model artifact to models/lightgbm_tuned.pkl')
